In [1]:
import pandas as pd
from difflib import SequenceMatcher
from tabulate import tabulate

In [2]:
df = pd.read_csv("../data/document_data_clean_filtered.csv", encoding="utf-8")
df

,ark,title_clean,date_clean,author_name_clean,author_type_clean,author_birth_clean,author_death_clean,publisher_name_clean,publisher_place_clean,type_clean,format,description,type,source,language,title,author,date,publisher
0,bpt6k6148781w,"Botanique.... SUPPL., T. 3",1783.0,"Lamarck, Jean-Baptiste de Monet de",person,1744.0,1829.0,"Panckoucke (H. Agasse, Vve Agasse)",Paris,monographie,"13 vol. in-4° , dont 5 de supplément",[Encyclopédie botanique (français)]\tCollectio...,printed monograph,"Bibliothèque nationale de France, département ...",fre,"Botanique.... SUPPL., T. 3 / par M. le chevali...","Lamarck, Jean-Baptiste de Monet de (1744-1829)...",1783-1817,"Panckoucke (H. Agasse, Vve Agasse) (Paris)"
1,bpt6k61498285,"Botanique.... SUPPL., T. 4",1783.0,"Lamarck, Jean-Baptiste de Monet de",person,1744.0,1829.0,"Panckoucke (H. Agasse, Vve Agasse)",Paris,monographie,"13 vol. in-4° , dont 5 de supplément",[Encyclopédie botanique (français)]\tCollectio...,printed monograph,"Bibliothèque nationale de France, département ...",fre,"Botanique.... SUPPL., T. 4 / par M. le chevali...","Lamarck, Jean-Baptiste de Monet de (1744-1829)...",1783-1817,"Panckoucke (H. Agasse, Vve Agasse) (Paris)"
2,bpt6k6156104p,"Botanique.... SUPPL., T. 5",1783.0,"Lamarck, Jean-Baptiste de Monet de",person,1744.0,1829.0,"Panckoucke (H. Agasse, Vve Agasse)",Paris,monographie,"13 vol. in-4° , dont 5 de supplément",[Encyclopédie botanique (français)]\tCollectio...,printed monograph,"Bibliothèque nationale de France, département ...",fre,"Botanique.... SUPPL., T. 5 / par M. le chevali...","Lamarck, Jean-Baptiste de Monet de (1744-1829)...",1783-1817,"Panckoucke (H. Agasse, Vve Agasse) (Paris)"
3,bpt6k6151194j,"Botanique.... SUPPL., T. 1",1783.0,"Lamarck, Jean-Baptiste de Monet de",person,1744.0,1829.0,"Panckoucke (H. Agasse, Vve Agasse)",Paris,monographie,"13 vol. in-4° , dont 5 de supplément",[Encyclopédie botanique (français)]\tCollectio...,printed monograph,"Bibliothèque nationale de France, département ...",fre,"Botanique.... SUPPL., T. 1 / par M. le chevali...","Lamarck, Jean-Baptiste de Monet de (1744-1829)...",1783-1817,"Panckoucke (H. Agasse, Vve Agasse) (Paris)"
4,bpt6k61522197,"Botanique.... SUPPL., T. 2",1783.0,"Lamarck, Jean-Baptiste de Monet de",person,1744.0,1829.0,"Panckoucke (H. Agasse, Vve Agasse)",Paris,monographie,"13 vol. in-4° , dont 5 de supplément",[Encyclopédie botanique (français)]\tCollectio...,printed monograph,"Bibliothèque nationale de France, département ...",fre,"Botanique.... SUPPL., T. 2 / par M. le chevali...","Lamarck, Jean-Baptiste de Monet de (1744-1829)...",1783-1817,"Panckoucke (H. Agasse, Vve Agasse) (Paris)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19028,bpt6k5001416j,Le Petit colon algérien,1883.0,NaN,NaN,NaN,NaN,Bureaux,Alger,presse quotidienne,NaN,"24 décembre 1883\t1883/12/24 (A6,N1990).\t",printed serial,"Bibliothèque nationale de France, département ...",fre,Le Petit colon algérien,NaN,1883-12-24,Bureaux (Alger)
19029,bpt6k3529169g,Le Voltaire,1883.0,NaN,NaN,NaN,NaN,Le Voltaire,Paris,presse quotidienne,NaN,"25 décembre 1883\t1883/12/25 (A6,N1999).\t",printed serial,"Bibliothèque nationale de France, département ...",fre,Le Voltaire / directeur Aurélien Scholl,NaN,1883-12-25,Le Voltaire (Paris)
19030,bpt6k60476669,Le Courrier de Saint-Pierre,1883.0,NaN,NaN,NaN,NaN,[s.n.] (Saint-Pierre,Réunion),presse scientifique,NaN,"29 décembre 1883\t1883/12/29 (A1,N87).\t\tAppa...",printed serial,"Bibliothèque nationale de France, département ...",fre,Le Courrier de Saint-Pierre : journal de l'arr...,NaN,1883-12-29,[s.n.] (Saint-Pierre (Réunion))
19031,bpt6k15513767,Le Mercure aptésien,1883.0,NaN,NaN,NaN,NaN,J.-S. Jean,Apt,presse quotidienne,NaN,"30 décembre 1883\t1883/12/30 (A45,N3222).\t\tA...",printed serial,"Archives éditeur le Mercure aptésien, Collecti...",fre,Le Mercure aptésien : journal de l'arrondissem...,NaN,1883-12-30,J.-S. Jean (Apt)


In [3]:
with open("../data/other/presse_quotidienne.txt", encoding='utf-8') as f:
    presse_quotidienne = f.readlines()
    presse_quotidienne = list(set([presse.strip() for presse in presse_quotidienne]))

In [4]:
def type_cleaner(type_clean, title):
    # Remove from consideration if not printed serial
    if type_clean != "printed serial":
        return "monographie"
    
    #check for "annuaires":
    if "annuaire" in title.lower():
        return "annuaire"
    
    #Check for médical / specialised key words
    specialized_word_list = ["médecin", "médic", "académie", "mycologique", "cosmos"]
    for word in specialized_word_list:
        if word.lower() in title.lower():
            return "presse scientifique"
    
    #check if in journal database
    for presse in presse_quotidienne:
        if presse.lower() in title.lower():
            return "presse quotidienne"
    
    official_word_list = ["comité de madagascar", "actes administratifs", "budget", "procès-verbaux", "comité de l'afrique française", "comité de l'asie française", "régence", "république", "ministère", "chambre des députés", "préfecture", "gouvernement", "président", "assemblée", "officiel", "conseil général", "session", "projets de lois"]
    for word in official_word_list:
        if word.lower() in title.lower():
            return "presse officielle"

    #Check for journal "Paris" without checking for every single title that has "Paris" on it:
    if title.lower() == "paris":
        return "presse quotidienne"
    
    #check if title contains words typically associated with quotidien presse:
    quotidien_word_list = ["politique", "théâtr", "prose", "almanach", "spirituel", "poète", "eglise", "express", "automobile", "jésus", "chemin" "missions", "paroiss", "dimanche", "chrét", "cathol", "annonce", "illustr", "élégan", "industri", "art moderne", "portefeuille", "évangélique", "syndic", "économie", "artist", "patri", "musical", "populaire", "sociali", "religi", "sport", "humoristique", "administration", "gazette", "judiciaire", "municipal", "dépêche", "colon", "républicain", "hebdomadaire", "quotidien", "démocratie"]
    for word in quotidien_word_list:
        if word.lower() in title.lower():
            return "presse quotidienne"
    
    return "presse scientifique"

In [5]:
df["type_cleaner"] = df.apply(lambda x: type_cleaner(x["type_clean"], x["title"]), axis=1)

In [7]:
#Publisher_name
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    df_presse = df[df["type_cleaner"] == "annuaire"]
    print(len(df_presse))
    print(tabulate(df_presse.sample(20)[["title"]], headers='keys', tablefmt='psql'))

0


ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
#Publisher_name
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    df_presse = df[df["type_clean"] == "printed serial"]
    print(tabulate(df_presse.sample(100)[["type_cleaner", "title"]], headers='keys', tablefmt='psql'))

+-------+---------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|       | type_cleaner        | title                                                                                                                                                                                                                                                                                  |
|-------+---------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 17123 | presse scientifique | Les Conférences...           